# 4. Build meta-features

**Input:** validation component predictions.

**Output:** one Parquet file containing the 19 inputs used by each residual stacker.

In [ ]:
# Step 1 - Imports and inputs

import os
import sys

import numpy as np
import pandas as pd
from tqdm import tqdm

sys.path.append(os.path.abspath("../../../")) ; from EPF import variables

HORIZONS = range(1, variables.HORIZON_COUNT+1)
META_FEATURES = [
    "base_l1", "base_l2", "spike_mae", "spike_q90", "dip_mae", "dip_q10",
    "spike_probability", "dip_probability", "spike_logit", "dip_logit",
    "base_l2_minus_l1", "spike_mae_positive_gap", "spike_q90_positive_gap",
    "dip_mae_negative_gap", "dip_q10_negative_gap", "weighted_spike_mae_gap",
    "weighted_spike_q90_gap", "weighted_dip_mae_gap", "weighted_dip_q10_gap",
]
component_predictions = pd.read_parquet(variables.VALIDATION_COMPONENT_PREDICTIONS_PATH)

In [ ]:
# Step 2 - Build the 19 features

def build_meta_features(horizon):
    prediction = {name: component_predictions[f"{name}_h{horizon}"].values for name in ["base_l1", "base_l2", "spike_probability", "spike_mae", "spike_q90", "dip_probability", "dip_mae", "dip_q10"]}
    base_l1 = prediction["base_l1"]
    spike_probability = np.clip(prediction["spike_probability"], 1e-6, 1-1e-6)
    dip_probability = np.clip(prediction["dip_probability"], 1e-6, 1-1e-6)
    spike_mae_gap = np.maximum(prediction["spike_mae"]-base_l1, 0)
    spike_q90_gap = np.maximum(prediction["spike_q90"]-base_l1, 0)
    dip_mae_gap = np.minimum(prediction["dip_mae"]-base_l1, 0)
    dip_q10_gap = np.minimum(prediction["dip_q10"]-base_l1, 0)

    values = {
        **prediction,
        "spike_probability": spike_probability,
        "dip_probability": dip_probability,
        "spike_logit": np.log(spike_probability/(1-spike_probability)),
        "dip_logit": np.log(dip_probability/(1-dip_probability)),
        "base_l2_minus_l1": prediction["base_l2"]-base_l1,
        "spike_mae_positive_gap": spike_mae_gap,
        "spike_q90_positive_gap": spike_q90_gap,
        "dip_mae_negative_gap": dip_mae_gap,
        "dip_q10_negative_gap": dip_q10_gap,
        "weighted_spike_mae_gap": spike_probability*spike_mae_gap,
        "weighted_spike_q90_gap": spike_probability*spike_q90_gap,
        "weighted_dip_mae_gap": dip_probability*dip_mae_gap,
        "weighted_dip_q10_gap": dip_probability*dip_q10_gap,
    }
    return {f"{name}_h{horizon}": values[name].astype(np.float32) for name in META_FEATURES}

In [ ]:
# Step 3 - Save

meta_features = {}
for horizon in tqdm(HORIZONS):
    meta_features.update(build_meta_features(horizon))

meta_features = pd.DataFrame(meta_features, index=component_predictions.index)
meta_features.index.name = "date"
meta_features.to_parquet(variables.VALIDATION_META_FEATURES_PATH)

display(meta_features.head())
print(meta_features.shape)
print(variables.VALIDATION_META_FEATURES_PATH)